# Modelo de Identificação de Fraudes

- Informações da base:
    - 'Time' (tempo): contém os segundos decorridos entre cada transação e a primeira transação no conjunto de dados. 
    - 'Amount' (valor): é o valor da transação 
    - 'Class' (classe): é a variável de resposta e assume valor 1 em caso de fraude e 0 caso contrário.
    - "Infelizmente, devido a questões de confidencialidade, não podemos fornecer os recursos originais e mais informações básicas sobre os dados. Características V1, V2, … V28 são os principais componentes obtidos com PCA"

In [ ]:
# Configuração do ambiente

# Garante que o notebook está na raiz do projeto
%cd .. 

# Verifica o diretório atual (Linux/Mac)
# !pwd  

# Verifica o diretório atual (Windows)
!cd 

In [ ]:
# Importações necessárias 

import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import datetime as dt
import xgboost as xgb

from scipy.stats import kstest, normaltest, anderson

from sklearn.preprocessing import PowerTransformer, MinMaxScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn import tree, metrics
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.model_selection import train_test_split, GridSearchCV
from imblearn.under_sampling import RandomUnderSampler, ClusterCentroids, NearMiss 
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.combine import SMOTEENN

from sklearn.metrics import (
    confusion_matrix, 
    accuracy_score, 
    precision_score, 
    recall_score,
    precision_recall_curve,
)

import warnings
from src.config.paths import RAW_DATA_PATH, PROCESSED_DATA_PATH
from src.config.auxiliares_ml import downcast_dataframe, testar_modelos_com_undersampling

warnings.filterwarnings('ignore')
sns.set_theme(palette="bright")

In [ ]:
# Importando a base

transacoes = pd.read_parquet(PROCESSED_DATA_PATH)

In [ ]:
# Visualizando estatísticas descritivas da base 

transacoes.describe()

### NOTA: Vamos melhorar a escala das features

In [ ]:
# Lista de colunas a serem normalizadas (excluindo 'Class')

colunas_para_normalizar = ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
                           'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
                           'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']

# Normalizando as colunas (min-max scaling)

transacoes[colunas_para_normalizar] = transacoes[colunas_para_normalizar] / transacoes[colunas_para_normalizar].max()

In [ ]:
# Visualizando estatísticas descritivas da base normalizada

transacoes.describe()

In [ ]:
# Separando X e y

X = transacoes.drop(columns='Class')
y = transacoes['Class']

In [ ]:
# Separando em treino e teste

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=0,stratify=y)

### XGBoost 

In [ ]:
# Criando o modelo XGBoost

modelo_xgb = xgb.XGBClassifier(
    max_depth=9,
    n_estimators=200,
    learning_rate=0.3,
    random_state=0,
    scale_pos_weight=(y.value_counts()[0] / y.value_counts()[1])  # Lida com desbalanceamento
)

# Treinando e avaliando o modelo

modelo_xgb.fit(X_train, y_train)

y_pred_xgb = modelo_xgb.predict(X_test)

# Calculando as métricas

acc_xgb = accuracy_score(y_test, y_pred_xgb)
prec_xgb = precision_score(y_test, y_pred_xgb)
rec_xgb = recall_score(y_test, y_pred_xgb)

# Vusualizando a matriz confusão

confusion_matrix(y_test, y_pred_xgb)

In [ ]:
# Utilizando as métricas de avaliação

print('Acurácia:', accuracy_score(y_test, y_pred_xgb))
print('Precisão:', precision_score(y_test, y_pred_xgb))
print('Recall:', recall_score(y_test, y_pred_xgb, pos_label=1))

### Random Forest

In [ ]:
# Utilizando o Random Forest

clf_RF = RandomForestClassifier(
    max_depth=7,  
    random_state=0,
    criterion='gini',
    n_estimators=100,  
    class_weight='balanced_subsample' 
).fit(X_train, y_train)

y_pred_RF = clf_RF.predict(X_test)

# Vusualizando a matriz confusão

confusion_matrix(y_test, y_pred_RF)

In [ ]:
# Utilizando as métricas de avaliação

print('Acurácia:', accuracy_score(y_test, y_pred_RF))
print('Precisão:', precision_score(y_test, y_pred_RF))
print('Recall:', recall_score(y_test, y_pred_RF))

### Regressão Logística

In [ ]:
# Utilizando a Regressão Logística

clfLog = LogisticRegression(random_state=0).fit(X_train, y_train)

# Calculando as métricas

y_predLog = clfLog.predict(X_test)
y_pred_probaLog = clfLog.predict_proba(X_test)[:,1]

# Vusualizando a matriz confusão

confusion_matrix(y_test, y_predLog)

In [ ]:
# Utilizando as métricas de avaliação

print('Acurácia:', accuracy_score(y_test, y_predLog))
print('Precisão:', precision_score(y_test, y_predLog))
print('Recall:', recall_score(y_test, y_predLog))

### OBSERVAÇÃO:

**Utilizamos a base desbalanceada e fizemos uma espécie de 'normalização' nos dados**.

- Utilizando os três melhores algoritmos até aqui, temos como desempenho:
  
  - **1º Lugar**: XGBoost
  - **2º Lugar**: Random Forest
  - **3º Lugar**: Regressão Logística